# 02 — Phase 2 QLoRA training (Colab / cloud GPU)

Fine-tunes a classifier head on top of `google/gemma-4-E4B-it` using QLoRA (bnb 4-bit nf4). Vision and audio towers are dropped right after load so steady-state memory is text-only.

**Hardware**: needs CUDA. Don't run on Apple Silicon — bitsandbytes nf4 is CUDA-only. Use `01_smoke_test.ipynb` for any local Mac exploration.

**Recommended Colab tier**: A100 (40 GB) or A10. T4 (16 GB) is borderline — even after dropping vision/audio, the text tower at 4-bit + LoRA + activations is tight.

Tracks issue: https://github.com/Charlemagne-Labs/gateguard-suite/issues/73

## 1. Clone repo and install

Run these as shell cells (`!`-prefixed) the first time only.

In [ ]:
# Skip if you've already cloned and installed in this Colab session.
!git clone https://github.com/Charlemagne-Labs/g4h.git
%cd g4h
!pip install -q -e ".[train]"

## 2. Authenticate to Hugging Face

`google/gemma-4-E4B-it` is gated. Accept the license at https://huggingface.co/google/gemma-4-E4B-it then paste a read token below.

In [ ]:
from huggingface_hub import login
login()  # paste token from https://huggingface.co/settings/tokens

## 3. Sanity checks

In [ ]:
import torch
import transformers, peft, accelerate, bitsandbytes

assert torch.cuda.is_available(), "This notebook requires CUDA."
print(f"torch:        {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"peft:         {peft.__version__}")
print(f"accelerate:   {accelerate.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")
print(f"GPU:          {torch.cuda.get_device_name(0)}")
print(f"GPU memory:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 4. Get the training data

We need `data/latest_ft_train_data.csv` (3259 rows, columns: `text`, `label`). Pick one path:

### Option A — manual upload (simplest)

1. Open the **Files** panel on the left side of Colab.
2. Navigate into `g4h/data/`.
3. Drag `latest_ft_train_data.csv` from your laptop into that directory.

Skip option B and run the verify cell below.

### Option B — pull from gateguard-suite (requires gh auth on Colab)

```python
!gh auth login --web --git-protocol https  # interactive
!gh repo clone Charlemagne-Labs/gateguard-suite -- --depth 1 --filter=blob:none --no-checkout /tmp/gg
!cd /tmp/gg && git sparse-checkout set data/latest_ft_train_data.csv && git checkout
!cp /tmp/gg/data/latest_ft_train_data.csv data/
```

In [ ]:
# Verify the data is in place
import pandas as pd
from collections import Counter

CSV_PATH = "data/latest_ft_train_data.csv"
df = pd.read_csv(CSV_PATH)
print(f"rows: {len(df)}, cols: {list(df.columns)}")
print(f"label distribution: {dict(Counter(df['label']))}")
assert {'text', 'label'}.issubset(df.columns), "CSV must have text and label columns"
print("\nFirst row text (truncated):")
print("  " + str(df.iloc[0]['text'])[:200])

## 5. QLoRA config + training

Defaults match the gateguard 270m-it baseline recipe. Change `cfg` fields if you want to sweep.

In [ ]:
from transformers import BitsAndBytesConfig
from src.train import run_training, TrainConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

cfg = TrainConfig(
    model_id="google/gemma-4-E4B-it",
    csv_path=CSV_PATH,
    out_dir="runs/gemma4-e4b-cls",
    val_ratio=0.10,
    test_ratio=0.10,
    max_length=256,
    epochs=3,
    batch=8,
    grad_accum=2,
    lr=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    lora_target_modules=("q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"),
    use_class_weights=True,
    seed=42,
)

metrics = run_training(cfg, bnb_config=bnb)
print()
print("Final val metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

## 6. Inspect saved artifacts

In [ ]:
import os
for root, dirs, files in os.walk(cfg.out_dir):
    level = root.replace(cfg.out_dir, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        size = os.path.getsize(os.path.join(root, f))
        print(f'{indent}  {f}  ({size:,} bytes)')

## 7. Package for download

Tarball the run directory so you can download it from Colab in one click. The output shape matches gateguard's `inference_config.json` schema, so this tarball can be uploaded to `charley-s3-teacher-model` as `teacher_model.tar.gz` and consumed by `Charlemagne-Labs/test_suite/phase2`.

In [ ]:
import tarfile
tarball = f"{cfg.out_dir.rstrip('/')}.tar.gz"
with tarfile.open(tarball, "w:gz") as t:
    t.add(cfg.out_dir, arcname=os.path.basename(cfg.out_dir))
print(f"Wrote {tarball} ({os.path.getsize(tarball):,} bytes)")
print("Right-click the file in the Colab Files panel and pick 'Download'.")